# VayuKavach AQI forecast models

Re-runs the rolling 12-month backtest from stored data and shows the per-day error table. Run from `backend/` with `DATABASE_URL` pointing at the database (`uv run --with jupyter jupyter lab`).

See `model-card.md` for what the models are and their limits.

In [ ]:
from datetime import UTC, datetime, timedelta

from app.db import SessionLocal
from app.forecast.model import backtest, summary
from app.forecast.serve import load_frame, today_ist

with SessionLocal() as session:
    df = load_frame(session)
today = today_ist(datetime.now(UTC))
df.groupby('district_id')['date'].agg(['min', 'max', 'count'])

In [ ]:
bt = backtest(df[df['date'] < today], end=today - timedelta(days=1), weeks=52)
s = summary(bt)
s

In [ ]:
bt.table.pivot(index='day', columns='method', values='mae').round(1)

In [ ]:
bt.table.pivot(index='day', columns='method', values='hit_rate').round(2)

## Reading the result

- Days 1–5 compare corrected CAMS with raw CAMS and tomorrow = today.
- Days 6–10 compare the outlook median with tomorrow = today and the city's month average.
- `coverage_80`: share of truths inside the outlook's 5–95% quantile range (target ≥ 80%).